# 2次元イジング模型：シミュレーションと厳密解の比較

このノートでは、モンテカルロシミュレーションで得られた結果を、ラーシュ・オンサガーによって導出された**2次元イジング模型の厳密解**と比較し、その精度と「有限サイズ効果」を検証します。

---

## 1. 厳密解（オンサガーの解）

### 転移温度 $T_c$
2次元正方格子における転移温度の厳密な値は以下の通りです：
$$ T_c = \frac{2J}{k_B \ln(1 + \sqrt{2})} \approx 2.269185 J $$

### 自発磁化 $M(T)$
無限系（$L \to \infty$）における自発磁化 $M$ は、楊振寧（C. N. Yang）によって以下のように与えられました：
$$ M(T) = \begin{cases} \left[ 1 - \left( \sinh \frac{2J}{k_B T} \right)^{-4} \right]^{1/8} & (T < T_c) \\ 0 & (T \ge T_c) \end{cases} $$

---

## 2. 比較用プログラムの実装

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def exact_magnetization(T, J=1.0):
    """オンサガーの厳密解（無限系の磁化）"""
    Tc = 2.0 * J / np.log(1.0 + np.sqrt(2.0))
    if T >= Tc:
        return 0.0
    else:
        m = (1.0 - (np.sinh(2.0 * J / T))**(-4))**(1.0/8.0)
        return m

# 比較用の温度範囲
T_range = np.linspace(1.5, 3.0, 100)
M_exact = [exact_magnetization(t) for t in T_range]
Tc_exact = 2.0 / np.log(1.0 + np.sqrt(2.0))

plt.figure(figsize=(8, 5))
plt.plot(T_range, M_exact, 'r-', label='Exact (Onsager/Yang)')
plt.axvline(x=Tc_exact, color='k', linestyle='--', label=f'Exact Tc ≈ {Tc_exact:.3f}')
plt.xlabel('Temperature T')
plt.ylabel('Magnetization M')
plt.title('Exact Solution of 2D Ising Model')
plt.legend()
plt.grid(True)
plt.show()

## 3. なぜシミュレーション結果と厳密解がズレるのか？

シミュレーションの結果を重ねると、特に $T_c$ 付近で厳密解（赤い線）から外れる様子が見えるはずです。これには2つの理由があります：

### (1) 有限サイズ効果 (Finite-Size Effect)
- **厳密解**: 無限に大きな系（$L = \infty$）を想定しています。そのため $T_c$ で磁化がスパッと 0 になります。
- **シミュレーション**: $L=16$ や $32$ といった小さな系です。このため、転移がなだらかになり、$T_c$ を超えても磁化が完全には 0 にならない「尾」を引く現象が見られます。

### (2) 統計誤差と緩和時間
- $T_c$ 付近ではスピンの向きが揃うかバラバラになるかの「せめぎ合い」が激しくなり、平衡状態に達するまで非常に時間がかかります（**臨界鈍化**）。
- 計算時間が足りないと、正しい統計平均が得られず誤差が大きくなります。

## 4. 検証：Lを大きくするとどうなるか？
有限サイズスケーリングの理論によれば、システムサイズ $L$ を大きくしていくと、シミュレーションの結果は一歩ずつ厳密解（無限系）へと近づいていきます。